In [16]:
import numpy as np

test = np.load('./data/results.npz')
test.files

['A',
 'B',
 'S',
 'V_exact',
 'V_average',
 'V_list',
 'evolution_average',
 'evolution_list',
 'single_step_error_before',
 'single_step_error_after',
 'total_error_before',
 'total_error_after',
 'N',
 'J',
 'h',
 'T',
 'r',
 'Heisenberg',
 'trials']

In [19]:
test['total_error_after']

array(0.79188983)

In [ ]:
# 示例：如何使用 gen_q_tuples()
# 约束：每个 qi 的下界为 K+1，上界为 q_0，且 q1+...+qj = s
# 请根据你的公式替换 s 的具体定义
def BCH_term(q, matrix_list):
    """计算 BCH 展开式中的 q 阶项"""
    if q == 1:
        return sum(matrix_list)
    elif q == 2:
        term = 0
        for i in range(len(matrix_list)):
            for j in range(len(matrix_list)):
                if i < j:
                    term += commutator(matrix_list[i], matrix_list[j])
        return term / 2
    elif q == 3:
        term = 0
        for i in range(len(matrix_list)):
            for j in range(len(matrix_list)):
                for k in range(len(matrix_list)):
                    if i < j < k:
                        term += commutator(
                            matrix_list[i], commutator(matrix_list[j], matrix_list[k])
                        )
        return term / 6
    else:
        raise NotImplementedError("Only BCH terms up to order 3 are implemented.")


def F_term(order, matrix_list, q_0, K=K):
    """计算 F 项的示例实现（演示如何枚举合法的 (q1,...,qj) 并累加对应贡献）。
    目前这里把每个合法元组计数（占位），你应将占位处替换为基于 q_tuple 与 matrix_list 的实际计算。"""
    term = 0
    # 最大可能的 j 长度
    max_j = order // (K + 1)
    for j_len in range(1, max_j + 1):
        continue
    return term

In [ ]:
import numpy as np

# Set the seed of rng first
rng = np.random.default_rng(seed=7)
J = 0.5
h = 1
N = 4
k = 2
g = 2 * (J + h)

# closed boundary condition
W_tot = J * (N - 1) + h * N

"""
Hamiltonian list for 1D Ising model:
[Z_1, X_1X_2, Z_2, X_2X_3 ,,, Z_{N-1}, X_{N-1}X_N, Z_{N}]
"""


# get Pauli H_l according to its index in the list:
def get_local_H(index):
    if index % 2 == 0:
        # even index: Z term
        site = index // 2
        H_l = np.kron(np.eye(2**site), np.kron(Z, np.eye(2 ** (N - site - 1))))
    else:
        # odd index: XX term
        site = index // 2
        H_l = np.kron(
            np.eye(2**site), np.kron(np.kron(X, X), np.eye(2 ** (N - site - 2)))
        )
    return H_l


# corresponding norm list:
norms = np.array([h, J] * (N - 1) + [h])


# intersected neighbors
def get_neighbor(S):
    neighbor = []
    for index in S:
        if index % 2 == 0:
            # even: Z, [index - 1, index, index + 1]
            start = max(0, index - 1)
            end = min(2 * N - 2, index + 1)
        else:
            # odd: XX, [index - 2, index - 1, index, index + 1, index + 2]
            start = max(0, index - 2)
            end = min(2 * N - 2, index + 2)

        for i in range(start, end + 1):
            neighbor.append(i)

    return list(set(neighbor))


def LCSample_debug(q, tol=1e-12):
    if q < 2:
        raise ValueError("Length q must be at least 2.")

    L = len(norms)
    p_tot = norms / W_tot

    print("L =", L)
    print("p_tot sum =", p_tot.sum())
    print("W_tot/(2*N*q*k*g) =", W_tot / (2 * N * q * k * g))

    l1 = int(rng.choice(np.arange(L), p=p_tot))
    print("l1 =", l1, "p(l1) =", p_tot[l1])

    r = rng.random()
    thresh0 = W_tot / (2 * N * q * k * g)
    print("step0 rand =", r, "pass prob =", thresh0)
    if r > thresh0:
        print("reject at step0")
        return 0

    S = [l1]
    C = 1j * get_local_H(l1) / norms[l1]
    A = get_neighbor([l1])

    for j in range(2, q + 1):
        W_A = np.sum(norms[A])
        p_A = norms[A] / W_A

        l_j = int(rng.choice(A, p=p_A))
        print(
            f"step{j} A size =",
            len(A),
            "W_A =",
            W_A,
            "l_j =",
            l_j,
            "p(l_j) =",
            p_A[A.index(l_j)],
        )

        r = rng.random()
        thresh = W_A / (q * k * g)
        print(f"step{j} rand =", r, "pass prob =", thresh)
        if r > thresh:
            print(f"reject at step{j}")
            return 0

        comm = commutator(1j * get_local_H(l_j), C)
        comm_norm = np.linalg.norm(comm, ord=2)
        print(f"step{j} comm_norm =", comm_norm)

        if comm_norm <= tol:
            print(f"reject at step{j} (comm_norm <= tol)")
            return 0

        C = comm / (2 * norms[l_j])
        S.append(l_j)
        A = get_neighbor(S)

    print("accept")
    return S, C


LCSample_debug(3)

In [ ]:
def lcsample(H_list, supports, q, N, k, g, rng=None, norm_ord="fro", tol=1e-12):
    """Light-cone Commutator Sampling (Algorithm 2).
    H_list: list of numpy arrays H_ell
    supports: list of sets of qubit indices (same length as H_list)
    q: length of sequence to sample
    N, k, g: system params from paper
    Returns (S, C) where S is list of indices (1-based like paper),
    C is the accumulated operator (matrix). Returns 0 if abort condition hits.
    Decomposition of C into alpha, P is left to caller if needed.
    """
    rng = rng or np.random.default_rng()
    if len(H_list) != len(supports):
        raise ValueError("H_list and supports must have same length")
    L = len(H_list)
    norms = np.array([np.linalg.norm(H, ord=norm_ord) for H in H_list], dtype=float)
    if (norms <= 0).any():
        raise ValueError("All H_l must have non-zero norm")
    W_tot = norms.sum()
    # Step 2: sample l1
    p_tot = norms / W_tot
    l1 = int(rng.choice(np.arange(L), p=p_tot))
    # Step 3: acceptance test
    proceed_prob = min(1.0, W_tot / (2 * N * q * k * g))
    if rng.random() > proceed_prob:
        return 0
    # Step 4: init
    S = [l1 + 1]  # store as 1-based index to match paper
    C = 1j * H_list[l1] / norms[l1]
    # Step 5: candidates intersecting support of l1
    supp_l1 = set(supports[l1])
    A = {ell for ell in range(L) if ell != l1 and supp_l1 & set(supports[ell])}
    # Loop
    for j in range(2, q + 1):
        if not A:
            return 0
        W_A = norms[list(A)].sum()
        if W_A <= 0:
            return 0
        probs_A = norms[list(A)] / W_A
        l_choices = list(A)
        l_j = int(rng.choice(l_choices, p=probs_A))
        proceed_prob = min(1.0, W_A / (q * k * g))
        if rng.random() > proceed_prob:
            return 0
        comm = commutator(1j * H_list[l_j], C)
        if np.linalg.norm(comm, ord=norm_ord) <= tol:
            return 0
        C = comm / (2 * norms[l_j])
        S.append(l_j + 1)
        supp_lj = set(supports[l_j])
        A.update(
            {ell for ell in range(L) if ell != l_j and supp_lj & set(supports[ell])}
        )
    return S, C


# 示例：针对 ising_hamiltonian 生成 supports（按构造已知）
def ising_supports(num_bits):
    supports = []
    # H1 terms: single-qubit X on each site
    for i in range(num_bits):
        supports.append({i})
    # H2 terms: Z Z on odd-even pairs (starting at 1)
    for i in range(1, num_bits - 1, 2):
        supports.append({i, i + 1})
    # H3 terms: Z Z on even-odd pairs (starting at 0)
    for i in range(0, num_bits - 1, 2):
        supports.append({i, i + 1})
    return supports


# 构造 H_list 对应 supports 的一个简单例子
def build_H_list_with_supports(N, J, h):
    H1, H2, H3, _ = ising_hamiltonian(N, J, h)
    H_list = []
    supports = []
    # match the order in ising_supports: N single-qubit, then odd-even, then even-odd
    for i in range(N):
        # extract single-qubit X term for site i
        H_single = np.kron(np.eye(2**i), np.kron(X, np.eye(2 ** (N - i - 1)))) * (-h)
        H_list.append(H_single)
        supports.append({i})
    for i in range(1, N - 1, 2):
        interaction = -J * np.kron(
            np.eye(2**i), np.kron(np.kron(Z, Z), np.eye(2 ** (N - i - 2)))
        )
        H_list.append(interaction)
        supports.append({i, i + 1})
    for i in range(0, N - 1, 2):
        interaction = -J * np.kron(
            np.eye(2**i), np.kron(np.kron(Z, Z), np.eye(2 ** (N - i - 2)))
        )
        H_list.append(interaction)
        supports.append({i, i + 1})
    return H_list, supports


# 示例使用
# H_list, supports = build_H_list_with_supports(N, J, h)
# rng = np.random.default_rng(7)
# sample = lcsample(H_list, supports, q=3, N=N, k=k, g=g, rng=rng)
# print(sample)